- Ingest data in data lakehouse 
- perform data quality checks and transform the data as requested -- silver_clean
- apply changes to the customers_data -- silver

In [0]:
CREATE OR REFRESH STREAMING TABLE bronze_customers
  COMMENT 'Raw customers data ingested from source system'
  TBLPROPERTIES ('quality' = 'bronze') AS
SELECT
  *,
  _metadata.file_path AS file_path,
  current_timestamp() AS ingestion_timestamp
FROM
  cloud_files(
    '/Volumes/circuitbox/landing/operational_data/customers/',
    'json',
    map('cloudFiles.inferColumnTypes', 'true')
  );

In [0]:
CREATE OR REFRESH STREAMING TABLE silver_customers_clean(
  CONSTRAINT valid_customer_id EXPECT(customer_id IS NOT NULL) ON VIOLATION FAIL UPDATE,
  CONSTRAINT valid_customer_name EXPECT(customer_name IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_telephone EXPECT(len(telephone) >= 10),
  CONSTRAINT valid_email EXPECT(email IS NOT NULL),
  CONSTRAINT valid_date_of_birth EXPECT(date_of_birth >= '1920-01-01')
)
  COMMENT 'Cleaned customers data'
  TBLPROPERTIES ('quality' = 'silver') AS
SELECT
  customer_id,
  customer_name,
  CAST(date_of_birth as DATE) as date_of_birth,
  telephone,
  email,
  CAST(created_date AS DATE) created_date
FROM
  STREAM(LIVE.bronze_customers)